# **Lung CT Classification & Knowledge Preservation System**
### **Hybrid Classifier (ConvNeXt-Tiny + Handcrafted 82-D Fusion) with Grad-CAM, LLM, Human-in-the-Loop & Case-Based Reasoning (CBR)**
---
* **Dataset**: Unified PSG-IQ Lung CT Cohort (1097 IQ-OTHNCCD images + 53 PSG IMSR development images + 17 external test images)
* **Models**: Genetic Programming (GP), XGBoost (Original & Fine-Tuned), Hybrid Classifier (Baseline, Second-Tuned, First-Tuned Champion @ **95.45% Accuracy**)
* **Explainability**: Grad-CAM visual attention saliency overlay targeting predicted classes
* **Decision Support**: Gemini LLM structured clinical diagnostic explanations
* **Human-in-the-Loop (HITL)**: Attending doctor verification & correction workflow
* **Case-Based Reasoning (CBR)**: Doctor-verified case base retrieval, similarity-weighted consensus voting & candidate retraining


## **1. Environment Setup & Reproducibility**


In [ ]:
import os
import sys
import json
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from pathlib import Path
from scipy import ndimage as ndi
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

from xgboost import XGBClassifier
from gplearn.genetic import SymbolicClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('ENVIRONMENT & HARDWARE CHECK')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Scikit-Learn    :', __import__('sklearn').__version__)
print('Device          :', device)


## **2. Dataset Paths & Split Configuration (PSG-IQ Cohort)**


In [ ]:
# Project directories
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'ipynb' else Path('.').resolve()
NEW_DATASET_DIR = PROJECT_ROOT / 'New Dataset' / 'PSG-IQ Lung CT Dataset'

IQ_DIR = NEW_DATASET_DIR / 'The IQ-OTHNCCD lung cancer dataset'
PSG_DIR = NEW_DATASET_DIR / 'PSG IMSR Dataset Testing'
PSG_EXT_TEST_DIR = PSG_DIR / 'Test'

MODELS_DIR = PROJECT_ROOT / 'backend' / 'models'
SAVED_MODELS_DIR = PROJECT_ROOT / 'backend' / 'models_saved'
SPLIT_DIR = MODELS_DIR / 'splits'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Normal', 'Benign', 'Malignant']
CLASS_TO_IDX = {'Normal': 0, 'Benign': 1, 'Malignant': 2}

print('Dataset Directory Exists:', NEW_DATASET_DIR.exists())
print('IQ-OTHNCCD Exists       :', IQ_DIR.exists())
print('PSG IMSR Exists         :', PSG_DIR.exists())
print('External Test Reserved  :', PSG_EXT_TEST_DIR.exists())


## **3. Dataset Compiling & Split Verification**


In [ ]:
# Load 1097 IQ-OTHNCCD Images
iq_records = []
for cname in CLASS_NAMES:
    folder = IQ_DIR / f'{cname} cases'
    if folder.exists():
        for f in folder.iterdir():
            if f.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                iq_records.append({
                    'filepath': str(f), 'filename': f.name,
                    'class': cname, 'label': CLASS_TO_IDX[cname], 'cohort': 'IQ-OTHNCCD'
                })
iq_df = pd.DataFrame(iq_records)

# Load 53 PSG IMSR Development Images
psg_folder_map = {'Normal': 'normal', 'Benign': 'benign', 'Malignant': 'maligancy'}
psg_records = []
for cname, sub in psg_folder_map.items():
    folder = PSG_DIR / sub
    if folder.exists():
        for f in folder.iterdir():
            if f.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                psg_records.append({
                    'filepath': str(f), 'filename': f.name,
                    'class': cname, 'label': CLASS_TO_IDX[cname], 'cohort': 'PSG-IMSR'
                })
psg_df = pd.DataFrame(psg_records)

# Stratified Split of IQ Dataset (70% Train, 20% Val, 10% Internal Test)
iq_train_df, iq_temp_df = train_test_split(iq_df, test_size=0.30, random_state=SEED, stratify=iq_df['label'])
iq_val_df, iq_int_test_df = train_test_split(iq_temp_df, test_size=1/3, random_state=SEED, stratify=iq_temp_df['label'])

# Combine 110 IQ Internal Test + 53 PSG IMSR into the Clinical Internal Test Set = 163 samples!
clinical_test_df = pd.concat([iq_int_test_df, psg_df], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('=' * 65)
print('DATASET SPLIT SUMMARY')
print('=' * 65)
print(f'Training Set (IQ)             : {len(iq_train_df)} images')
print(f'Validation Set (IQ)           : {len(iq_val_df)} images')
print(f'Combined Clinical Test (IQ+PSG): {len(clinical_test_df)} images')
print(f'Total Development Dataset     : {len(iq_train_df) + len(iq_val_df) + len(clinical_test_df)} images')
print('Untouched External Test Set   : 17 images (Reserved in Test/ folder)')


## **4. Classical Lung Field Segmentation & Preprocessing**


In [ ]:
def segment_lung(image_input):
    if isinstance(image_input, (str, Path)):
        original = cv2.imread(str(image_input), cv2.IMREAD_GRAYSCALE)
    else:
        original = image_input.copy()
    if original is None:
        raise ValueError('Unable to read image')

    h, w = original.shape
    blurred = cv2.GaussianBlur(original, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    binary = (binary > 0).astype(np.uint8)

    labeled, _ = ndi.label(binary)
    border_labels = set(labeled[0, :].tolist()) | set(labeled[-1, :].tolist()) | set(labeled[:, 0].tolist()) | set(labeled[:, -1].tolist())
    border_labels.discard(0)
    border_cleared = np.where(np.isin(labeled, list(border_labels)), 0, binary).astype(np.uint8) if border_labels else binary

    opened = cv2.morphologyEx(border_cleared, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    labeled2, num2 = ndi.label(opened)
    if num2 == 0:
        return original, np.zeros((h, w), dtype=np.uint8), original.copy()

    sizes = ndi.sum(opened, labeled2, range(1, num2 + 1))
    candidate_labels = [i + 1 for i, s in enumerate(sizes) if s > 0.003 * h * w]
    if not candidate_labels:
        return original, np.zeros((h, w), dtype=np.uint8), original.copy()

    candidate_labels = sorted(candidate_labels, key=lambda lab: sizes[lab - 1], reverse=True)[:2]
    mask = np.isin(labeled2, candidate_labels).astype(np.uint8)
    mask = ndi.binary_fill_holes(mask).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=2)
    mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    mask = ndi.binary_fill_holes(mask).astype(np.uint8)

    roi = original.copy()
    roi[mask == 0] = 0
    return original, mask, roi

print('Classical lung segmentation pipeline ready.')


## **5. 82-D Fused Representation (Deep ConvNeXt + Handcrafted Features)**


In [ ]:
def extract_gp_features_from_norm(norm_img, enhanced_img):
    hog_feat = hog(norm_img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True).astype(np.float32)
    radius = 2
    points = 8 * radius
    lbp = local_binary_pattern(enhanced_img, points, radius, method='uniform')
    n_bins = points + 2
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    lbp_hist = lbp_hist.astype(np.float32) / (lbp_hist.sum() + 1e-8)
    intensity = np.array([
        np.mean(norm_img), np.std(norm_img), np.min(norm_img), np.max(norm_img),
        np.percentile(norm_img, 10), np.percentile(norm_img, 25), np.percentile(norm_img, 50),
        np.percentile(norm_img, 75), np.percentile(norm_img, 90)
    ], dtype=np.float32)
    return np.concatenate([hog_feat, lbp_hist, intensity])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT).to(device)
convnext.eval()

# Load fitted preprocessing objects
prep_objects = joblib.load(MODELS_DIR / 'preprocessing_objects.joblib')
print('Loaded Preprocessing Objects:', list(prep_objects.keys()))


## **6. Hybrid Classifier Architecture (Tuned MLP on 82-D Fused Features)**


In [ ]:
class TunedMLPClassifier(nn.Module):
    def __init__(self, input_dim=82, num_classes=3, dropout1=0.15, dropout2=0.10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout1),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.network(x)

# Load Champion First-Tuned Model (Rank 1 @ 95.45% Accuracy)
hybrid_champion = TunedMLPClassifier(input_dim=82, num_classes=3).to(device)
ckpt_p = MODELS_DIR / 'hybrid_models' / 'hybrid_classifier_best.pt'
ckpt = torch.load(ckpt_p, map_location=device)
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    hybrid_champion.load_state_dict(ckpt['model_state_dict'])
else:
    hybrid_champion.load_state_dict(ckpt)
hybrid_champion.eval()
print('Loaded Hybrid Classifier — First-Tuned (Champion): Ready for inference!')


## **7. Final Model Comparison Benchmark Table**


In [ ]:
metrics_file = MODELS_DIR / 'model_metrics.json'
with open(metrics_file, 'r', encoding='utf-8') as f:
    metrics_data = json.load(f)

table_df = pd.DataFrame(metrics_data['comparison_table'])
print('=' * 85)
print('LUNG CT CLASSIFICATION — FINAL MODEL COMPARISON BENCHMARK')
print('=' * 85)
display_cols = ['rank', 'model', 'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']
styled_df = table_df[display_cols].copy()
styled_df.columns = ['Rank', 'Model', 'Accuracy (%)', 'Balanced Accuracy (%)', 'Macro Precision (%)', 'Macro Recall (%)', 'Macro F1 (%)', 'Weighted F1 (%)']
print(styled_df.to_string(index=False))


## **8. Grad-CAM Visual Attention Saliency Overlay**


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(self, img_tensor):
        self.model.zero_grad()
        out = self.model(img_tensor)
        target_class = torch.argmax(out[0])
        out[0, target_class].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True)).squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, int(target_class)

target_layer = convnext.features[-1]
gradcam_engine = GradCAM(convnext, target_layer)
print('Grad-CAM XAI Engine initialized on ConvNeXt target layer.')


## **9. Gemini LLM Clinical Decision Support Explanation**


In [ ]:
def generate_clinical_explanation(case_id, predicted_class, confidence, probabilities, focus_pct=88.5):
    api_key = os.environ.get('GEMINI_API_KEY')
    if api_key:
        try:
            from google import genai
            client = genai.Client(api_key=api_key)
            prompt = f'''Provide a clinical radiological decision explanation for Case {case_id}:
Predicted Class: {predicted_class} (Confidence: {confidence*100:.1f}%)
Probabilities: Normal={probabilities['Normal']*100:.1f}%, Benign={probabilities['Benign']*100:.1f}%, Malignant={probabilities['Malignant']*100:.1f}%
Heatmap Focus inside Lung: {focus_pct}%
Use standard clinical headings without technical CV model names.'''
            resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
            if resp and resp.text:
                return resp.text.strip()
        except Exception as e:
            pass
    return f'''1. AI DIAGNOSTIC FINDING:
The system analyzed the CT scan for {case_id} and identified the primary finding as {predicted_class} with an assigned confidence score of {confidence*100:.1f}%.

2. PROBABILITY ASSESSMENT:
• Normal: {probabilities['Normal']*100:.1f}%
• Benign: {probabilities['Benign']*100:.1f}%
• Malignant: {probabilities['Malignant']*100:.1f}%

3. VISUAL HEATMAP ATTENTION:
Grad-CAM indicates {focus_pct}% attention concentrated within the segmented lung parenchyma.

4. CLINICAL DISCLAIMER:
AI-estimated probabilities are clinical decision support aids and do not constitute a definitive medical diagnosis.'''

print('Clinical Decision Support Explanation module ready.')


## **10. Case-Based Reasoning (CBR) & Doctor-in-the-Loop Knowledge Base**


In [ ]:
CASE_BASE_FILE = SAVED_MODELS_DIR / 'doctor_verified_case_base.json'

def query_cbr_cases(query_feat_82, top_k=5, min_similarity=0.75):
    if not CASE_BASE_FILE.exists():
        return {'status': 'empty', 'matches': [], 'recommendation': None}
    with open(CASE_BASE_FILE, 'r', encoding='utf-8') as f:
        cases = json.load(f)
        
    q_vec = np.asarray(query_feat_82, dtype=np.float32).reshape(-1)
    q_norm = np.linalg.norm(q_vec)
    if q_norm <= 1e-12:
        return {'status': 'invalid', 'matches': [], 'recommendation': None}
        
    scores = []
    for c in cases:
        feat = c.get('fused_features')
        if not feat or len(feat) != 82:
            continue
        v = np.asarray(feat, dtype=np.float32).reshape(-1)
        sim = float(np.dot(q_vec, v) / (q_norm * np.linalg.norm(v) + 1e-12))
        sim = float(np.clip(sim, -1.0, 1.0))
        if sim >= min_similarity:
            scores.append({
                'case_id': c.get('case_id'),
                'image_filename': c.get('image_filename'),
                'doctor_verified_class': c.get('doctor_verified_class'),
                'similarity': round(sim, 4),
                'similarity_pct': round(sim * 100, 1),
                'doctor_notes': c.get('doctor_notes', '')
            })
            
    scores.sort(key=lambda x: x['similarity'], reverse=True)
    top_matches = scores[:top_k]
    
    votes = {'Normal': 0.0, 'Benign': 0.0, 'Malignant': 0.0}
    for m in top_matches:
        votes[m['doctor_verified_class']] += m['similarity']
        
    tot = sum(votes.values())
    dist = {k: round(v / tot, 4) if tot > 0 else 0.333 for k, v in votes.items()}
    rec = max(dist, key=dist.get) if top_matches else None
    
    return {
        'status': 'success',
        'total_cases_in_base': len(cases),
        'cases_meeting_threshold': len(scores),
        'top_matches': top_matches,
        'recommendation': rec,
        'confidence': dist.get(rec, 0.0),
        'vote_distribution': dist
    }

print('CBR Knowledge Preservation & Retrieval System active.')
if CASE_BASE_FILE.exists():
    with open(CASE_BASE_FILE, 'r') as f:
        print(f'Total doctor-verified cases preserved in case base: {len(json.load(f))}')


## **11. Complete Pipeline Execution (Hybrid + Grad-CAM + LLM + HITL + CBR)**


In [1]:
# Test with a sample from the clinical test cohort
sample_case = clinical_test_df.iloc[0]
sample_path = sample_case['filepath']
print('=' * 70)
print('END-TO-END CLINICAL DIAGNOSTIC PREDICTION DEMO')
print('=' * 70)
print('Sample Image    :', sample_case['filename'])
print('True Verified   :', sample_case['class'])

gray = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
orig, mask, roi = segment_lung(gray)
roi_pil = Image.fromarray(roi).convert('RGB')
img_t = val_transform(roi_pil).unsqueeze(0).to(device)

norm_img = cv2.resize(roi, (224, 224), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
enhanced_img = cv2.resize(roi, (224, 224), interpolation=cv2.INTER_AREA)

# 1. Feature extraction
with torch.no_grad():
    feat_768 = convnext.avgpool(convnext.features(img_t)).flatten(1).detach().cpu().numpy()
dp_32 = prep_objects['deep_pca'].transform(prep_objects['deep_scaler'].transform(feat_768))

raw_hc = extract_gp_features_from_norm(norm_img, enhanced_img)[:50]
hc_50 = prep_objects['feature_scaler'].transform([raw_hc])

fused_82 = prep_objects['hybrid_scaler'].transform(np.hstack([hc_50, dp_32]))

# 2. Hybrid Classification
with torch.no_grad():
    logits = hybrid_champion(torch.tensor(fused_82, dtype=torch.float32).to(device))
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

pred_idx = int(np.argmax(probs))
predicted_class = CLASS_NAMES[pred_idx]
conf = float(probs[pred_idx])

probs_dict = {'Normal': float(probs[0]), 'Benign': float(probs[1]), 'Malignant': float(probs[2])}

print(f'\n[1] HYBRID CLASSIFIER (First-Tuned @ 95.45% Acc):')
print(f'Predicted Finding : {predicted_class} ({conf*100:.2f}% Confidence)')
print(f'Probabilities     : Normal={probs[0]*100:.1f}%, Benign={probs[1]*100:.1f}%, Malignant={probs[2]*100:.1f}%')

# 3. CBR Case Retrieval
cbr_res = query_cbr_cases(fused_82[0].tolist(), top_k=5)
print(f'\n[2] CASE-BASED REASONING (CBR):')
print(f'CBR Consensus Rec : {cbr_res.get("recommendation")} ({cbr_res.get("confidence", 0)*100:.1f}% Confidence)')
print(f'Agreement with AI : {cbr_res.get("recommendation") == predicted_class}')
print(f'Top Matched Historical Cases:')
for m in cbr_res.get('top_matches', [])[:3]:
    print(f'  • {m["case_id"]} | Verified: {m["doctor_verified_class"]} | Match: {m["similarity_pct"]} %')

# 4. LLM Narrative
print(f'\n[3] CLINICAL DECISION SUPPORT EXPLANATION:')
print(generate_clinical_explanation('CASE-DEMO-001', predicted_class, conf, probs_dict))


NameError: name 'clinical_test_df' is not defined